In [ ]:
# ============================================
# Phase 0 - Bag 生成主脚本
# 功能：读取标注CSV → 切分Bags → 生成弱标签 → 保存bags.pkl
# ============================================
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple

# ─── 1. 配置 ──────────────────────────────────────────────────────────────
CONFIG = {
    "data_root": "D:/Project_Github/audio_click_mil/data",
    "output_dir": "D:/Project_Github/audio_click_mil/results/phase0",
    "bags_output": "D:/Project_Github/audio_click_mil/data/bags.pkl",
    "audio_sr": 48000,          # 目标采样率 48kHz
    "bag_duration": 30,         # 秒
    "recording_duration": 1800, # 30分钟 = 1800秒
    "overlap": 0,               # 无重叠
}

# 计算每条录音的bag数量
BAGS_PER_RECORDING = CONFIG["recording_duration"] // CONFIG["bag_duration"]  # 60个

# ─── 2. 辅助函数 ───────────────────────────────────────────────────────────
def get_bag_id(file_num: int, bag_idx: int) -> str:
    """生成Bag ID: F{file_num:03d}_B{bag_idx:04d}"""
    return f"F{file_num:03d}_B{bag_idx:04d}"

def get_bag_time_range(bag_idx: int, bag_dur: int = 30) -> Tuple[float, float]:
    """计算bag的时间范围 (start, end) 秒"""
    start = bag_idx * bag_dur
    end = start + bag_dur
    return start, end

def check_event_in_bag(event_start: float, event_end: float, 
                       bag_start: float, bag_end: float) -> bool:
    """判断事件是否与bag时间范围有重叠"""
    return event_start < bag_end and event_end > bag_start

# ─── 3. 读取所有标注文件 ───────────────────────────────────────────────────
print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始读取标注文件...")

data_root = Path(CONFIG["data_root"])
whistle_df = pd.read_csv(data_root / "WhistleParameters-clean.csv")
click_df = pd.read_csv(data_root / "ClickTrains.csv")
burst_df = pd.read_csv(data_root / "BurstPulseTrains.csv")
buzz_df = pd.read_csv(data_root / "BuzzTrains.csv")
results_df = pd.read_csv(data_root / "Results.csv")

# 统一列名（根据实际CSV调整）
# 哨声文件编号列
whistle_file_col = "Original Audio File (No.)  "  # 注意可能有空格
# 脉冲文件编号列
pulse_file_col = "Ori_file_num(No.)"

# 清理文件编号（转为int）
whistle_df[whistle_file_col] = whistle_df[whistle_file_col].fillna(0).astype(int)
click_df[pulse_file_col] = click_df[pulse_file_col].fillna(0).astype(int)
burst_df[pulse_file_col] = burst_df[pulse_file_col].fillna(0).astype(int)
buzz_df[pulse_file_col] = buzz_df[pulse_file_col].fillna(0).astype(int)

print(f"  ✓ 哨声标注: {len(whistle_df)} 条")
print(f"  ✓ Click标注: {len(click_df)} 条")
print(f"  ✓ Burst标注: {len(burst_df)} 条")
print(f"  ✓ Buzz标注: {len(buzz_df)} 条")
print(f"  ✓ 录音文件: {len(results_df)} 条")

# ─── 4. 生成所有Bags ───────────────────────────────────────────────────────
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 开始生成Bags...")
print(f"  参数: {CONFIG['bag_duration']}秒/bag, 无重叠, {BAGS_PER_RECORDING} bags/录音")

all_bags = []
bag_id_list = []

# 获取所有有标注的文件编号（1-35）
all_file_nums = sorted(results_df["Original audio_file (No.)"].dropna().astype(int).unique())

for file_num in all_file_nums:
    # 提取该文件的所有标注
    file_whistles = whistle_df[whistle_df[whistle_file_col] == file_num]
    file_clicks = click_df[click_df[pulse_file_col] == file_num]
    file_bursts = burst_df[burst_df[pulse_file_col] == file_num]
    file_buzzes = buzz_df[buzz_df[pulse_file_col] == file_num]
    
    # 生成该文件的60个bags
    for bag_idx in range(BAGS_PER_RECORDING):
        bag_id = get_bag_id(file_num, bag_idx)
        bag_start, bag_end = get_bag_time_range(bag_idx, CONFIG["bag_duration"])
        
        # 检查是否有哨声
        has_whistle = False
        whistle_types = []
        for _, row in file_whistles.iterrows():
            w_start = row["Whistle Begins (s)"]
            w_end = w_start + 2  # 假设哨声持续2秒（或从CSV读取）
            if check_event_in_bag(w_start, w_end, bag_start, bag_end):
                has_whistle = True
                whistle_types.append(row.get("Type", "Unknown"))
        
        # 检查是否有脉冲（Click + Burst + Buzz）
        has_pulse = False
        for df in [file_clicks, file_bursts, file_buzzes]:
            for _, row in df.iterrows():
                p_start = row["Train_start(s)"]
                p_end = row["Train_end(s)"]
                if check_event_in_bag(p_start, p_end, bag_start, bag_end):
                    has_pulse = True
                    break
            if has_pulse:
                break
        
        # 构建bag字典
        bag = {
            "bag_id": bag_id,
            "file_num": file_num,
            "bag_idx": bag_idx,
            "time_start": bag_start,
            "time_end": bag_end,
            "whistle_label": 1 if has_whistle else 0,
            "pulse_label": 1 if has_pulse else 0,
            "whistle_types": list(set(whistle_types)) if whistle_types else [],
            "audio_sr": CONFIG["audio_sr"],
            "audio_samples": CONFIG["bag_duration"] * CONFIG["audio_sr"],  # 30 * 48000 = 1440000
        }
        
        all_bags.append(bag)
        bag_id_list.append(bag_id)

print(f"  ✓ 共生成 {len(all_bags)} 个 bags")

# ─── 5. 统计弱标签分布 ─────────────────────────────────────────────────────
print(f"\n[{datetime.now().strftime('%H:%M:%S')}] 弱标签分布统计:")

whistle_count = sum(1 for b in all_bags if b["whistle_label"] == 1)
pulse_count = sum(1 for b in all_bags if b["pulse_label"] == 1)
both_count = sum(1 for b in all_bags if b["whistle_label"] == 1 and b["pulse_label"] == 1)
neg_count = sum(1 for b in all_bags if b["whistle_label"] == 0 and b["pulse_label"] == 0)

print(f"  - 仅哨声 (whistle=1, pulse=0): {whistle_count - both_count}")
print(f"  - 仅脉冲 (whistle=0, pulse=1): {pulse_count - both_count}")
print(f"  - 哨声+脉冲 (whistle=1, pulse=1): {both_count}")
print(f"  - 背景噪声 (whistle=0, pulse=0): {neg_count}")
print(f"  - 哨声总计: {whistle_count}")
print(f"  - 脉冲总计: {pulse_count}")

# ─── 6. 保存 bags.pkl ──────────────────────────────────────────────────────
output_path = Path(CONFIG["bags_output"])
output_path.parent.mkdir(exist_ok=True, parents=True)

with open(output_path, "wb") as f:
    pickle.dump(all_bags, f)

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] ✓ bags.pkl 已保存至: {output_path}")

# ─── 7. 生成 dataset_summary.csv ───────────────────────────────────────────
summary_df = pd.DataFrame(all_bags)
summary_path = Path(CONFIG["output_dir"]) / "dataset_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"  ✓ dataset_summary.csv 已保存至: {summary_path}")

# ─── 8. 保存执行日志 ───────────────────────────────────────────────────────
log_lines = [
    f"Phase 0 - Bag 生成完成 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "=" * 60,
    f"总Bag数: {len(all_bags)}",
    f"每条录音Bag数: {BAGS_PER_RECORDING}",
    f"Bag时长: {CONFIG['bag_duration']}秒",
    f"采样率: {CONFIG['audio_sr']}Hz",
    "",
    "弱标签分布:",
    f"  哨声Bag: {whistle_count} ({whistle_count/len(all_bags)*100:.1f}%)",
    f"  脉冲Bag: {pulse_count} ({pulse_count/len(all_bags)*100:.1f}%)",
    f"  混合Bag: {both_count} ({both_count/len(all_bags)*100:.1f}%)",
    f"  负样本Bag: {neg_count} ({neg_count/len(all_bags)*100:.1f}%)",
]

log_path = Path(CONFIG["output_dir"]) / "02_bag_generation_log.txt"
log_path.write_text("\n".join(log_lines), encoding='utf-8')
print(f"  ✓ 日志已保存至: {log_path}")

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Phase 0 完成！")